# Logistic regression: probabilities, labels, and decisions

**Question.** Can wine chemistry distinguish the recorded cultivars, and what does a classification score mean operationally?

We will examine class balance, derive the logistic link and log-loss idea, fit a trusted reference through the package, inspect confusion patterns and confidence, and identify what a student implementation must contribute. This multiclass example teaches the workflow; a real product must define the consequences of each action and error.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay

from dsm_project.config import load_config
from dsm_project.data import load_project_dataset
from dsm_project.experiments import fit_supervised_candidate

config = load_config("configs/example.toml")
dataset = load_project_dataset(config)
task = dataset.tasks["classification"]
frame = dataset.observations

print(task)
display(frame[[*task.feature_columns[:4], task.target_column]].head())

## Begin with prevalence and separation

Accuracy can look impressive when one class dominates. We therefore inspect counts before choosing metrics. A two-feature view can reveal overlap and scale, but the fitted model uses the full declared feature vector and the projection cannot establish global linear separability.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
counts = frame[task.target_column].value_counts().sort_index()
counts.plot.bar(ax=axes[0], color="#7570b3", rot=0)
axes[0].set(title="Cultivar prevalence", xlabel="cultivar", ylabel="sample count")

for label, group in frame.groupby(task.target_column):
    axes[1].scatter(group["alcohol"], group["color_intensity"], label=f"class {label}", alpha=0.75)
axes[1].set(title="A two-feature view", xlabel="alcohol", ylabel="color intensity")
axes[1].legend()
plt.tight_layout();

## From an affine score to a probability

For binary classification, one neuron forms a preactivation

$$
z=w^\top x+b
$$

and maps it through the sigmoid

$$
\sigma(z)=\frac{1}{1+e^{-z}},
\qquad 0<\sigma(z)<1.
$$

With event label $y\in\{0,1\}$, minimizing binary cross-entropy

$$
-\frac{1}{n}\sum_i\left[y_i\log p_i+(1-y_i)\log(1-p_i)\right]
$$

penalizes confident wrong probabilities strongly. Its weight gradient has the familiar form $X^\top(p-y)/n$. Multiclass logistic regression generalizes the construction with one score per class and normalized probabilities.

A probability model, a label rule, and an operational action are different layers. A threshold converts probability to a label; costs and policy convert evidence to action.

In [ ]:
z = np.linspace(-8, 8, 400)
sigmoid = 1.0 / (1.0 + np.exp(-z))

plt.figure(figsize=(8, 4))
plt.plot(z, sigmoid, color="#e7298a", linewidth=2)
plt.axhline(0.5, linestyle="--", color="gray")
plt.axvline(0.0, linestyle="--", color="gray")
plt.xlabel("preactivation z")
plt.ylabel("sigmoid(z)")
plt.title("The sigmoid maps every real score into (0, 1)")
plt.tight_layout();

In [ ]:
result = fit_supervised_candidate(
    dataset,
    config,
    task_name="classification",
    model_name="logistic_regression",
)
print("Implementation:", result.model_name)
print(result.metric_name, result.metric_value)
print("Train/holdout rows:", result.train_rows, result.test_rows)

## Classification evidence is structured, not one number

Balanced accuracy gives every true class equal weight. The confusion matrix retains which classes are confused. Predicted confidence asks a different question: when the model is very sure, is it usually right? A professional analysis also studies calibration, class-conditional errors, uncertainty, and the real cost of each mistake.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ConfusionMatrixDisplay.from_predictions(
    result.observed,
    result.predictions,
    ax=axes[0],
    colorbar=False,
)
axes[0].set_title("Held-out confusion matrix")

probabilities = result.estimator.predict_proba(result.holdout_features)
confidence = probabilities.max(axis=1)
correct = result.predictions == result.observed
axes[1].hist(confidence[correct], bins=10, alpha=0.7, label="correct", color="#1b9e77")
axes[1].hist(confidence[~correct], bins=10, alpha=0.7, label="incorrect", color="#d95f02")
axes[1].set(title="Confidence is not the same as correctness", xlabel="maximum class probability", ylabel="holdout count")
axes[1].legend()
plt.tight_layout();

display(
    pd.crosstab(
        pd.Series(result.observed, name="observed"),
        pd.Series(result.predictions, name="predicted"),
        margins=True,
    )
)

## Conclusion, limits, and student implementation

The reference path demonstrates the evaluation contract, not the required student algorithm. Your team should implement stable sigmoid or softmax calculations, cross-entropy, gradients, fitted class handling, probability prediction, and invalid-input behavior under `src/dsm_project/algorithms/student/`. Register the estimator and run it through the same package evaluation boundary.

For the project, define the event or classes in domain language; compare against majority and cost-aware baselines; justify the metric; examine calibration and error slices; and state how probabilities would—and would not—support a human or automated decision. Never infer that a high holdout score makes an action safe.